In [ ]:
from pathlib import Path
import os
import sys

project_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "pyproject.toml").is_file())
sys.path.insert(0, str(project_root))
from experiments.paths import GENERATED, PROJECT_ROOT

run_directory = GENERATED / 'soay/age_size/data'
run_directory.mkdir(parents=True, exist_ok=True)
os.chdir(run_directory)


# Baseline Soay Sheep IBM

This notebook reproduces the ungulate life cycle, retains the final completed simulation year, saves a canonical random sample of 3,000 sheep, and saves a 1,000-row subset for downstream GP work. Vital rates are parameterized by GLMs. GLM parameters are taken from Ellner et al. 2016: https://github.com/ipmbook/first-edition/tree/master/Rcode/c6

In [ ]:
from pathlib import Path; import sys; sys.path.insert(0, str(Path.cwd().parent.parent))
import numpy as np
import pandas as pd
from experiments.soay.common.ipm_utils import simulate_sheep_ibm

In [ ]:
m_par_true = {
    "surv_int": -17.0,
    "surv_slope": 6.68,
    "surv_age": -0.334,

    "growth_int": 1.27,
    "growth_slope": 0.612,
    "growth_age": -0.00724,
    "growth_noise": 0.0787,

    "repr_int": -7.88,
    "repr_slope": 3.11,
    "repr_age": -0.078,

    "recr_int": 1.11,
    "recr_age": 0.184,

    "rcsz_int": 0.362,
    "rcsz_slope": 0.709,
    "rcsz_noise": 0.159,

    "female_prob": 0.5,
}

In [ ]:
rng = np.random.default_rng(270875)
result = simulate_sheep_ibm(m_par_true, n_years=400, init_pop_size=500, rng=rng, max_population=5000,use_age=True, retain="final")
final_year_data = result["data"]

if len(final_year_data) < 3000:
    raise ValueError(f"The last completed year contains only {len(final_year_data)} sheep; 3,000 are required")
print(f"Last Completed Year: {result['last_year']}")
print(f"Last-Year Population Before Sampling: {len(final_year_data)}")
print(f"Next Population Size: {result['next_population_size']}")

In [ ]:
rows = rng.choice(len(final_year_data), 3000, replace=False)
baseline_3000 = final_year_data.iloc[rows].sort_values("z").reset_index(drop=True)
baseline_1000 = baseline_3000.sample(1000, replace=False, random_state=53241986).sort_values("z").reset_index(drop=True)
baseline_3000.to_csv("baseline_ibm_data_agesize_3000.csv", index=False)
baseline_1000.to_csv("baseline_ibm_data_agesize_1000.csv", index=False)

In [ ]:
baseline_1000.age.value_counts()